In [3]:
!pip install -q transformers peft accelerate bitsandbytes datasets trl safetensors pandas lxml

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 31.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 531.0/531.0 kB 36.8 MB/s eta 0:00:00


In [4]:
import os, re, csv, time, random
import xml.etree.ElementTree as ET
import numpy as np
import pandas as pd
import torch

from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig

print('Torch:', torch.__version__)
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

Torch: 2.10.0+cu128
CUDA: True
GPU: Tesla T4


In [6]:
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)

CONFIG = {
    'model_name': 'Qwen/Qwen2.5-Coder-1.5B-Instruct',
    'max_seq_length': 512,
    'lora_r': 8,
    'lora_alpha': 16,
    'lora_dropout': 0.05,
    'learning_rate': 1e-4,
    'num_train_epochs': 1,
    'per_device_train_batch_size': 1,
    'gradient_accumulation_steps': 2,
    'warmup_ratio': 0.03,
    'weight_decay': 0.01,
    'logging_steps': 20,
    'save_steps': 1000,
    'max_train_samples': 2000,
    'svg_max_len': 2500,
    'output_dir': '/kaggle/working/fast',
}
CONFIG

{'model_name': 'Qwen/Qwen2.5-Coder-1.5B-Instruct',
 'max_seq_length': 512,
 'lora_r': 8,
 'lora_alpha': 16,
 'lora_dropout': 0.05,
 'learning_rate': 0.0001,
 'num_train_epochs': 1,
 'per_device_train_batch_size': 1,
 'gradient_accumulation_steps': 2,
 'warmup_ratio': 0.03,
 'weight_decay': 0.01,
 'logging_steps': 20,
 'save_steps': 1000,
 'max_train_samples': 2000,
 'svg_max_len': 2500,
 'output_dir': '/kaggle/working/fast'}

## 1. Load Data

In [7]:
TRAIN_PATH = '/kaggle/input/datasets/rebeccaschenn/dlmidterm/train.csv'
TEST_PATH  = '/kaggle/input/datasets/rebeccaschenn/dlmidterm/test.csv'

train_df = pd.read_csv(TRAIN_PATH, engine='python', quotechar='"',
                        quoting=csv.QUOTE_ALL, on_bad_lines='skip')
test_df  = pd.read_csv(TEST_PATH,  engine='python', quoting=csv.QUOTE_MINIMAL)

print('train columns:', train_df.columns.tolist())
print('train shape  :', train_df.shape)
print('test shape   :', test_df.shape)
train_df.head(2)

train columns: ['id', 'prompt', 'svg']
train shape  : (50000, 3)
test shape   : (1000, 2)


,id,prompt,svg
0,fd61e324e0cec5c11f337d0bfe2858c8,The image features two orange squares with a m...,"<svg xmlns=""http://www.w3.org/2000/svg"" viewBo..."
1,999b3d4d5a860725bf9528910b5612f3,A simple smiley face with a wide open mouth an...,"<svg xmlns=""http://www.w3.org/2000/svg"" viewBo..."


## 2. Helper Functions

In [8]:
ALLOWED_TAGS = {
    'svg','g','path','rect','circle','ellipse','line','polyline','polygon',
    'defs','use','symbol','clipPath','mask','linearGradient','radialGradient',
    'stop','text','tspan','title','desc','style','pattern','marker','filter',
}
DISALLOWED_ATTRS_RE = re.compile(
    r'\s+(?:filling|onclick|onload|onmouseover|xmlns:xlink)=["\'][^"\']*("|\')',
    re.IGNORECASE
)

def normalize_svg(svg_text):
    if not svg_text: return ''
    svg_text = DISALLOWED_ATTRS_RE.sub('', svg_text)
    for attr in ('width', 'height', 'viewBox'):
        svg_text = re.sub(rf'(<svg\b[^>]*?)\b{attr}=["\'][^"\']*("|\')', r'\1', svg_text)
    svg_text = re.sub(
        r'(<svg\b)',
        '<svg xmlns="http://www.w3.org/2000/svg" width="256" height="256" viewBox="0 0 256 256"',
        svg_text, count=1
    )
    svg_text = re.sub(r'(xmlns="[^"]*")(?=.*xmlns="[^"]*")', '', svg_text)
    svg_text = re.sub(r'\s+', ' ', svg_text).strip()
    return svg_text

def has_disallowed_tags(svg_text):
    try:
        root = ET.fromstring(svg_text)
        for elem in root.iter():
            tag = elem.tag
            if isinstance(tag, str):
                local = tag.split('}')[-1] if '}' in tag else tag
                if local not in ALLOWED_TAGS: return True
        return False
    except Exception: return True

def is_valid_svg(svg_text):
    if not svg_text: return False
    try:
        root = ET.fromstring(svg_text)
        tag = root.tag.split('}')[-1] if '}' in root.tag else root.tag
        return tag == 'svg'
    except ET.ParseError: return False

print('Helper functions defined.')

Helper functions defined.


## 3. Data Subset Analysis & Sampling

50k rows is too large to XML-parse in full on Colab T4. We do a fast string pre-filter on all 50k rows, then only run the slow normalize+parse step on a 2× sample pool.

In [9]:
# ══════════════════════════════════════════════════════════════════════
# DATA SUBSET ANALYSIS & SAMPLING
# 50k SVG examples is too large to clean in full on Colab T4 (~5-10 min).
# Strategy: fast pre-filter on raw strings → sample 2x target → full clean.
# ══════════════════════════════════════════════════════════════════════

print(f'Raw train rows: {len(train_df)}')

# ── Step 1: Quick string-level filter (no XML parsing, very fast) ──────
quick_mask = (
    train_df['prompt'].notna() &
    train_df['svg'].notna() &
    (train_df['prompt'].astype(str).str.strip() != '') &
    (train_df['svg'].astype(str).str.strip() != '') &
    (train_df['svg'].astype(str).str.lower().str.startswith('<svg')) &
    (train_df['svg'].astype(str).str.len() <= CONFIG['svg_max_len']) &
    (train_df['prompt'].astype(str).str.len() <= 300)
)
quick_pool = train_df[quick_mask].reset_index(drop=True)
print(f'Rows passing quick filter: {len(quick_pool)} / {len(train_df)}')

# ── Step 2: SVG length distribution (inform max_seq_length choice) ────
svg_lens    = quick_pool['svg'].astype(str).str.len()
prompt_lens = quick_pool['prompt'].astype(str).str.len()
print(f'\nSVG length   — median:{svg_lens.median():.0f}  p90:{svg_lens.quantile(0.9):.0f}  p95:{svg_lens.quantile(0.95):.0f}  max:{svg_lens.max()}')
print(f'Prompt length — median:{prompt_lens.median():.0f}  p95:{prompt_lens.quantile(0.95):.0f}')

# ── Step 3: Sample 2× target before expensive XML cleaning ────────────
POOL_SIZE = min(CONFIG['max_train_samples'] * 2, len(quick_pool))
print(f'\nSampling {POOL_SIZE} rows for full XML cleaning...')
sample_pool = quick_pool.sample(n=POOL_SIZE, random_state=SEED).reset_index(drop=True)

# ── Step 4: Full clean on sample pool only ─────────────────────────────
df = sample_pool[['id', 'prompt', 'svg']].copy()
df['prompt'] = df['prompt'].astype(str).str.strip()
df['svg']    = df['svg'].astype(str).str.strip()
df['svg']    = df['svg'].apply(normalize_svg)
df = df[df['svg'].apply(is_valid_svg)].reset_index(drop=True)
df = df[~df['svg'].apply(has_disallowed_tags)].reset_index(drop=True)
df = df.drop_duplicates(subset=['svg']).reset_index(drop=True)
print(f'Usable rows after full cleaning: {len(df)}')

# ── Step 5: Final training sample ─────────────────────────────────────
train_small_df = df.sample(
    n=min(CONFIG['max_train_samples'], len(df)),
    random_state=SEED
).reset_index(drop=True)
print(f'Final training set: {len(train_small_df)} rows')
train_small_df.head(2)

Raw train rows: 50000
Rows passing quick filter: 28483 / 50000

SVG length   — median:1265  p90:2210  p95:2351  max:2500
Prompt length — median:101  p95:213

Sampling 4000 rows for full XML cleaning...
Usable rows after full cleaning: 3993
Final training set: 2000 rows


,id,prompt,svg
0,0c0b2bf9767155fec19d7af47d868689,The image features a solid black bell icon on ...,"<svg width=""256"" height=""256"" viewBox=""0 0 256..."
1,d8178a77-8104-4537-a8b6-a7fd1689db7b,A simple icon of a school building with a cent...,"<svg width=""256"" height=""256"" viewBox=""0 0 256..."


## 4. System Prompt

In [10]:
SYSTEM_PROMPT = (
    'You are an SVG code generator. '
    'Given a text description, output ONLY valid SVG code with '
    'width="256" height="256" viewBox="0 0 256 256". '
    'Do not include explanations or markdown code fences.'
)
print('System prompt set.')

System prompt set.


## 5. Load Model & LoRA

In [11]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,  # fp16 — required for T4
    bnb_4bit_use_double_quant=True,
)
tokenizer = AutoTokenizer.from_pretrained(CONFIG['model_name'], use_fast=True, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

model = AutoModelForCausalLM.from_pretrained(
    CONFIG['model_name'],
    quantization_config=bnb_config,
    device_map='auto',
    torch_dtype=torch.float16,  # ✅ must match bnb_4bit_compute_dtype
    trust_remote_code=True,
)
model.config.use_cache = False
model = prepare_model_for_kbit_training(model)
print('Loaded:', CONFIG['model_name'])

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Loaded: Qwen/Qwen2.5-Coder-1.5B-Instruct


In [12]:
lora_config = LoraConfig(
    r=CONFIG['lora_r'], lora_alpha=CONFIG['lora_alpha'],
    lora_dropout=CONFIG['lora_dropout'],
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    bias='none', task_type=TaskType.CAUSAL_LM,
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 9,232,384 || all params: 1,552,946,688 || trainable%: 0.5945


## 6. Build Dataset

In [13]:
# def make_chat_text(row):
#     messages = [
#         {'role': 'system',    'content': SYSTEM_PROMPT},
#         {'role': 'user',      'content': row['prompt']},
#         {'role': 'assistant', 'content': row['svg']},
#     ]
#     return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)

def make_chat_text(row):
    return f"Prompt: {row['prompt']}\nSVG:\n{row['svg']}"

train_small_df['text'] = train_small_df.apply(make_chat_text, axis=1)
print(train_small_df['text'].iloc[0][:600])
print(f'Avg text length: {train_small_df["text"].str.len().mean():.0f} chars')

Prompt: The image features a solid black bell icon on a white background.
SVG:
<svg width="256" height="256" viewBox="0 0 256 256" xmlns="http://www.w3.org/2000/svg" ><path fill="currentColor" fill-opacity="1.0" d="M159.10000610351562 134.8000030517578 L146.10000610351562 116.30000305175781 L146.10000610351562 86.30000305175781 A50.0 50.0 0.0 0 0 113.0999984741211 39.29999923706055 L113.0999984741211 36.599998474121094 A17.0 17.0 0.0 0 0 96.0999984741211 20.0 A16.200000762939453 16.200000762939453 0.0 0 0 80.0 36.599998474121094 L80.0 39.400001525878906 A50.0 50.0 0.0 0 0 46.0 86.4000015258789
Avg text length: 1397 chars


In [14]:
sample_texts = train_small_df['text'].sample(min(200, len(train_small_df)), random_state=SEED).tolist()
token_lengths = [len(tokenizer(t, truncation=False)['input_ids']) for t in sample_texts]
truncated = sum(1 for l in token_lengths if l > CONFIG['max_seq_length'])
print(f'Median tokens : {int(np.median(token_lengths))}')
print(f'95th pct      : {int(np.percentile(token_lengths, 95))}')
print(f'Max tokens    : {max(token_lengths)}')
print(f'Truncated (>{CONFIG["max_seq_length"]}): {truncated}/{len(sample_texts)} ({truncated/len(sample_texts)*100:.1f}%)')

Median tokens : 1028
95th pct      : 2182
Max tokens    : 2341
Truncated (>512): 147/200 (73.5%)


In [15]:
train_dataset = Dataset.from_pandas(train_small_df[['text']], preserve_index=False)
print(f'Dataset size: {len(train_dataset)}')

Dataset size: 2000


## 7. Training

In [16]:
sft_config = SFTConfig(
    output_dir=CONFIG['output_dir'],
    num_train_epochs=CONFIG['num_train_epochs'],
    per_device_train_batch_size=CONFIG['per_device_train_batch_size'],
    gradient_accumulation_steps=CONFIG['gradient_accumulation_steps'],
    learning_rate=CONFIG['learning_rate'],
    warmup_ratio=CONFIG['warmup_ratio'],
    weight_decay=CONFIG['weight_decay'],
    bf16=True,
    logging_steps=CONFIG['logging_steps'],
    save_strategy='steps',
    save_steps=CONFIG['save_steps'],
    max_grad_norm=0.3,
    lr_scheduler_type='cosine',
    optim='paged_adamw_8bit',
    report_to='none',
    seed=SEED,
    remove_unused_columns=False,
    max_length=CONFIG['max_seq_length'],
    dataset_text_field='text',
    packing=False,
)
trainer = SFTTrainer(
    model=model, args=sft_config,
    train_dataset=train_dataset,
    processing_class=tokenizer,
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Adding EOS to train dataset:   0%|          | 0/2000 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/2000 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/2000 [00:00<?, ? examples/s]

In [17]:
train_result = trainer.train()
print(train_result)

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss
20,0.927065
40,0.699875
60,0.654382
80,0.627073
100,0.542779
120,0.657249
140,0.552483
160,0.560836
180,0.494159
200,0.594367


TrainOutput(global_step=1000, training_loss=0.5459864540100098, metrics={'train_runtime': 5292.1139, 'train_samples_per_second': 0.378, 'train_steps_per_second': 0.189, 'total_flos': 7478701045327872.0, 'train_loss': 0.5459864540100098})


In [18]:
os.makedirs(CONFIG['output_dir'], exist_ok=True)
trainer.save_model(CONFIG['output_dir'])
tokenizer.save_pretrained(CONFIG['output_dir'])
print(f'Saved to: {CONFIG["output_dir"]}')

Saved to: /kaggle/working/fast


## 8. Inference

In [5]:
# load model

from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

base_model_name = "Qwen/Qwen2.5-Coder-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(base_model_name)

model = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)
model = PeftModel.from_pretrained(
    model,
    "/kaggle/working/fast"
)

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

ValueError: Can't find 'adapter_config.json' at '/kaggle/working/fast'

In [19]:
# CRITICAL: re-enable KV cache after training (gradient_checkpointing forces it off during train)
model.eval()
model.config.use_cache = True
if hasattr(model, 'gradient_checkpointing_disable'):
    model.gradient_checkpointing_disable()
print('Model in eval mode, cache enabled.')

Model in eval mode, cache enabled.


In [1]:
SVG_REGEX = re.compile(r'(<svg\b[\s\S]*?</svg>)', flags=re.IGNORECASE)

def extract_svg(text):
    if not text:
        return ''
    for tok in ['<|im_end|>', '<|endoftext|>']:
        idx = text.find(tok)
        if idx != -1:
            text = text[:idx]
    match = SVG_REGEX.search(text)
    if match:
        return match.group(1).strip()
    start = text.lower().find('<svg')
    if start != -1:
        partial = text[start:].strip()
        if not partial.lower().endswith('</svg>'):
            partial += '</svg>'
        return partial
    return ''

def fallback_svg():
    return (
        '<svg xmlns="http://www.w3.org/2000/svg" width="256" height="256" viewBox="0 0 256 256">'
        '<rect x="0" y="0" width="256" height="256" fill="white"/>'
        '<circle cx="128" cy="128" r="64" fill="#cccccc"/>'
        '</svg>'
    )

def generate_svg(prompt, max_new_tokens=180):
    prompt = prompt.strip()[:120]
    chat_text = f"Prompt: {prompt}\nSVG:\n"

    inputs = tokenizer(chat_text, return_tensors='pt').to(model.device)
    input_len = inputs['input_ids'].shape[1]

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
            use_cache=True,
        )

    decoded = tokenizer.decode(output_ids[0][input_len:], skip_special_tokens=False)
    svg = normalize_svg(extract_svg(decoded))

    used_fallback = (
        (not is_valid_svg(svg)) or
        has_disallowed_tags(svg) or
        (len(svg) > 15900)
    )

    if used_fallback:
        svg = fallback_svg()

    return svg, decoded, used_fallback

NameError: name 're' is not defined

In [21]:
for p in ['a simple blue bird icon',
          'a curved arrow in a square',
          'five horizontal lines of varying thickness']:
    svg, _, fb = generate_svg(p, max_new_tokens=400)
    print(f'[fallback={fb}] valid={is_valid_svg(svg)} len={len(svg)}')
    print(f'  {svg[:200]}')
    print('-'*60)

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


[fallback=True] valid=True len=199
  <svg xmlns="http://www.w3.org/2000/svg" width="256" height="256" viewBox="0 0 256 256"><rect x="0" y="0" width="256" height="256" fill="white"/><circle cx="128" cy="128" r="64" fill="#cccccc"/></svg>
------------------------------------------------------------
[fallback=False] valid=True len=271
  <svg width="256" height="256" viewBox="0 0 256 256" xmlns="http://www.w3.org/2000/svg"> <rect x="10" y="10" width="180" height="180" fill="#f0f0f0"/> <path d="M100,10 L100,190 A40,40 0 0 1 140,150 L16
------------------------------------------------------------
[fallback=False] valid=True len=402
  <svg xmlns="http://www.w3.org/2000/svg" width="256" height="256" viewBox="0 0 256 256" > <rect x="10" y="10" width="180" height="30" fill="#4CAF50"/> <rect x="10" y="40" width="180" height="30" fill="
------------------------------------------------------------


## 9. Generate Submissions

In [ ]:
results, fallback_count = [], 0

test_df_small = test_df.copy()   # 先全量；如果还慢就改成 test_df.iloc[:100]

for i, row in test_df_small.iterrows():
    svg, _, fb = generate_svg(row['prompt'], max_new_tokens=220)

    if fb:
        fallback_count += 1

    results.append({
        'id': row['id'],
        'svg': svg
    })

    if i < 3:
        print(f'[{i}] fb={fb} len={len(svg)}')
        print(f'  prompt: {row["prompt"][:80]}')
        print(f'  svg   : {svg[:200]}')
        print()

    if (i + 1) % 20 == 0:
        print(f"Processed {i+1}/{len(test_df_small)} | fallback={fallback_count}")

submission_df = pd.DataFrame(results)
print(f"Total:{len(submission_df)} | Fallback:{fallback_count} ({fallback_count/len(submission_df)*100:.1f}%)")

submission_df.to_csv('/kaggle/working/submission.csv', index=False)
print("Saved to /kaggle/working/submission.csv")

[0] fb=True len=199
  prompt: firewood stack cut logs wood with leaf illustration.
  svg   : <svg xmlns="http://www.w3.org/2000/svg" width="256" height="256" viewBox="0 0 256 256"><rect x="0" y="0" width="256" height="256" fill="white"/><circle cx="128" cy="128" r="64" fill="#cccccc"/></svg>

[1] fb=True len=199
  prompt: The image shows five horizontal lines of varying thicknesses and lengths, arrang
  svg   : <svg xmlns="http://www.w3.org/2000/svg" width="256" height="256" viewBox="0 0 256 256"><rect x="0" y="0" width="256" height="256" fill="white"/><circle cx="128" cy="128" r="64" fill="#cccccc"/></svg>

[2] fb=True len=199
  prompt: A stylized icon depicting a curved arrow within a square shape.
  svg   : <svg xmlns="http://www.w3.org/2000/svg" width="256" height="256" viewBox="0 0 256 256"><rect x="0" y="0" width="256" height="256" fill="white"/><circle cx="128" cy="128" r="64" fill="#cccccc"/></svg>

Processed 20/1000 | fallback=17
Processed 40/1000 | fallback=33
Processed 60/10

In [ ]:
valid_count = submission_df['svg'].apply(is_valid_svg).sum()
print(f'Valid          : {valid_count}/{len(submission_df)} ({valid_count/len(submission_df)*100:.1f}%)')
print(f'Disallowed tags: {submission_df["svg"].apply(has_disallowed_tags).sum()}')
print(f'Over 16000 chars: {(submission_df["svg"].str.len() > 16000).sum()}')

In [ ]:
submission_df.to_csv('/kaggle/working/submission.csv', index=False)
print('Saved: /kaggle/working/submission.csv')
submission_df.head()